In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import torchvision
from torchvision.ops import batched_nms

import time

import cv2
import numpy as np
import os
import glob as glob
from PIL import Image

import albumentations as A
from albumentations.pytorch import ToTensorV2

import random

from collections import Counter

from tqdm import tqdm

import warnings

warnings.filterwarnings("ignore")

In [2]:
print("Torch version:",torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("CPU Count:", os.cpu_count())

Torch version: 2.2.1+cu121
CUDA available: True
CUDA version: 12.1
GPU count: 1
Device name: NVIDIA GeForce RTX 2060 with Max-Q Design
CPU Count: 12


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [5]:
BASE_DATASET_PATH = '../../../datasets/KITTI/dataset/ablation_study'
train_data_path = f'{BASE_DATASET_PATH}/train/images/'
train_lbl_path = f'{BASE_DATASET_PATH}/train/labels/'

valid_data_path = f'{BASE_DATASET_PATH}/valid/images/'
valid_lbl_path = f'{BASE_DATASET_PATH}/valid/labels/'

In [6]:
def seed_everything(seed=42):
    import os, random, numpy as np, torch

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [7]:
# KITTI Dataset * Don't Care class is ignored!
CLASSES = [ 
    'Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting', 'Cyclist', 'Tram', 'Misc'
]

colors = ['#ffffff','#3a3b7b','#6a6ecf','#8ca351','#fff100','#ff00ff','#833c39','#e598a0']

NUM_CLASSES = len(CLASSES)
NUM_WORKERS =  4
BATCH_SIZE = 5
VAL_BATCH_SIZE = BATCH_SIZE * 2
RESIZE_TO = 640
EPOCHS = 30
WARMUP_EPOCHS = 3

LEARNING_RATE = 5e-4

BASE_LR = LEARNING_RATE
WEIGHT_DECAY = 1e-4

CONF_THRESHOLD = 0.5

MAP_IOU_THRESH = 0.5
NMS_IOU_THRESH = 0.45

PIN_MEMORY = True
SAVE_MODEL = True
LOAD_MODEL = False
AMP = True
DEBUG = False
ACCUMULATE = 4

S = [RESIZE_TO // 32, RESIZE_TO // 16]

ANCHORS = [
    # LARGE OBJECT SCALE (S=20)
    [
        (0.0737, 0.1407),
        (0.1173, 0.2590),
        (0.2270, 0.4612),
    ],
    # SMALL OBJECT SCALE (S=40)
    [
        (0.0223, 0.0665),
        (0.0420, 0.0937),
        (0.0294, 0.2288),
    ],
]



In [8]:
SCALE = 1.1
train_transforms = A.Compose(
    [
        A.LongestMaxSize(max_size=int(RESIZE_TO * SCALE)),

        A.PadIfNeeded(
            min_height=int(RESIZE_TO * SCALE),
            min_width=int(RESIZE_TO * SCALE),
            border_mode=cv2.BORDER_CONSTANT,
        ),

        A.RandomCrop(
            width=RESIZE_TO,
            height=RESIZE_TO,
        ),

        A.ShiftScaleRotate(
            rotate_limit=10,
            scale_limit=0.05,
            shift_limit=0.05,
            p=0.5,
            border_mode=cv2.BORDER_CONSTANT,
        ),

        A.HorizontalFlip(p=0.5),

        A.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05,
            p=0.3
        ),

        A.Normalize(
            mean=[0, 0, 0],
            std=[1, 1, 1],
            max_pixel_value=255,
        ),

        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        min_visibility=0.4,
        label_fields=[],
    ),
)

test_transforms = A.Compose(
    [
        A.LongestMaxSize(max_size=RESIZE_TO),
        A.PadIfNeeded(
            min_height=RESIZE_TO, min_width=RESIZE_TO, border_mode=cv2.BORDER_CONSTANT
        ),
        A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255,),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format="yolo", min_visibility=0.4, label_fields=[]),
)

In [9]:
""" 
Information about architecture config:
"B" indicating a residual block
"S" is for scale prediction block
"U" is for upsampling the feature map
"""
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 4],
    (128, 3, 2),
    ["B", 6],
    (256, 3, 2),
    ["B", 8],
    (512, 3, 2),
    ["B", 8],
    (1024, 3, 2),
    ["B", 6],
    (512, 1, 1),
    (1024, 3, 1),
    "S",
    (256, 1, 1),
    "U",
    (256, 1, 1),
    (512, 3, 1),
    "S",
]

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.use_bn_act = bn_act

    def forward(self, x):
        if self.use_bn_act:
            return self.leaky(self.bn(self.conv(x)))
        else:
            return self.conv(x)


class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()
        self.layers = nn.ModuleList()
        for repeat in range(num_repeats):
            self.layers += [
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels, kernel_size=3, padding=1),
                )
            ]

        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x)
            else:
                x = layer(x)

        return x


class ScalePrediction(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            CNNBlock(
                2 * in_channels, 3 * (num_classes + 5), bn_act=False, kernel_size=1
            ),
        )
        
        self.num_classes = num_classes

    def forward(self, x):
        return (
            self.pred(x)
            .reshape(x.shape[0], 3, self.num_classes + 5, x.shape[2], x.shape[3])
            .permute(0, 1, 3, 4, 2)
        )

class SiStNet(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []
        route_connections = []
        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1)
                route_connections.pop()

        return outputs

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels
        
        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(
                    CNNBlock(
                        in_channels,
                        out_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=1 if kernel_size == 3 else 0,
                    )
                )
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(ResidualBlock(in_channels, num_repeats=num_repeats))

            elif isinstance(module, str):
                if module == "S":
                    layers += [
                        ResidualBlock(in_channels, use_residual=False, num_repeats=1),
                        CNNBlock(in_channels, in_channels // 2, kernel_size=1),
                        ScalePrediction(in_channels=in_channels // 2, num_classes=self.num_classes),
                    ]
                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2))
                    in_channels = in_channels * 3

        return layers

if __name__ == "__main__": 
    model = SiStNet(num_classes=NUM_CLASSES)

In [10]:
def iou_width_height(boxes1, boxes2):

    intersection = torch.min(boxes1[..., 0], boxes2[..., 0]) * torch.min(
        boxes1[..., 1], boxes2[..., 1]
    )

    union = (
        boxes1[..., 0] * boxes1[..., 1]
        + boxes2[..., 0] * boxes2[..., 1]
        - intersection
    )

    return intersection / (union + 1e-9)


In [11]:
def intersection_over_union(boxes_preds, boxes_labels, box_format="midpoint"):

    if box_format == "midpoint":
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2

        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    elif box_format == "corners":
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]

        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    else:
        raise ValueError(f"Unsupported box_format: {box_format}")

    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    intersection = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

    box1_area = torch.abs(
        (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    )

    box2_area = torch.abs(
        (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    )

    union = box1_area + box2_area - intersection

    return intersection / (union + 1e-9)

In [12]:
def mean_average_precision(
    pred_boxes,
    true_boxes,
    epoch,
    num_classes,
    iou_threshold=MAP_IOU_THRESH,
    conf_threshold=CONF_THRESHOLD,
    box_format="midpoint",
    eps=1e-9,
):
    classes_ap = []

    unique_images = set()

    class_tp = torch.zeros(num_classes, device=DEVICE)
    class_fp = torch.zeros(num_classes, device=DEVICE)
    class_fn = torch.zeros(num_classes, device=DEVICE)

    images_per_class = torch.zeros(num_classes, device=DEVICE)
    instances_per_class = torch.zeros(num_classes, device=DEVICE)

    total_tp = 0
    total_fp = 0
    total_gt = 0

    # ===============================
    # Group GT boxes
    # ===============================
    gt_by_class_img = {}

    for gt in true_boxes:
        img_id, cls = gt[0], int(gt[1])

        unique_images.add(img_id)

        gt_by_class_img.setdefault(cls, {})
        gt_by_class_img[cls].setdefault(img_id, [])
        gt_by_class_img[cls][img_id].append(
            torch.tensor(gt[3:])
        )

        instances_per_class[cls] += 1

    # ===============================
    # Per-class evaluation
    # ===============================
    for c in range(num_classes):

        if c not in gt_by_class_img:
            continue

        ground_truths = gt_by_class_img[c]
        images_per_class[c] = len(ground_truths)
        
        detections = [
            [d[0], d[1], d[2], torch.tensor(d[3:])]
            for d in pred_boxes
            if d[1] == c and d[2] >= conf_threshold
        ]

        detections.sort(key=lambda x: x[2], reverse=True)
        
        gt_used = {
            img: torch.zeros(len(bboxes), device=DEVICE)
            for img, bboxes in ground_truths.items()
        }

        gt_tensors = {
            img: torch.stack([
                b if torch.is_tensor(b) else torch.tensor(b)
                for b in bboxes
            ]).to(DEVICE)
            for img, bboxes in ground_truths.items()
        }

        TP = torch.zeros(len(detections))
        FP = torch.zeros(len(detections))

        total_true_bboxes = sum(len(v) for v in ground_truths.values())
        total_gt += total_true_bboxes

        # ===============================
        # Match detections
        # ===============================
        for det_idx, det in enumerate(detections):
            img_id = det[0]

            if img_id not in ground_truths:
                FP[det_idx] = 1
                continue
                
            det_box = det[3].to(DEVICE)
            
            gts = ground_truths[img_id]
            # ------------------------------------------------
            # EARLY EXIT 1 — NO GT
            # ------------------------------------------------
            if len(gts) == 0:
                FP[det_idx] = 1
                continue

            # ------------------------------------------------
            # EARLY EXIT 2 — SINGLE GT
            # ------------------------------------------------
            if len(gts) == 1:
        
                gt_box = gts[0].to(DEVICE)
        
                best_iou = float(
                    intersection_over_union(
                        det_box,
                        gt_box,
                        box_format=box_format,
                    )
                )
                best_gt_idx = 0

            else:
                # ------------------------------------------------
                # VECTOR IoU
                # ------------------------------------------------
                gts_tensor = gt_tensors[img_id]
        
                ious = intersection_over_union(
                    det_box.unsqueeze(0),
                    gts_tensor,
                    box_format=box_format,
                )
        
                best_iou, best_gt_idx = ious.max(dim=0)
                best_iou = float(best_iou)
                best_gt_idx = int(best_gt_idx)

            # ------------------------------------------------
            # MATCH DECISION
            # ------------------------------------------------
            if best_iou >= iou_threshold:
        
                if gt_used[img_id][best_gt_idx] == 0:
                    TP[det_idx] = 1
                    gt_used[img_id][best_gt_idx] = 1
                else:
                    FP[det_idx] = 1
            else:
                FP[det_idx] = 1
        
        # ===============================
        # Precision-Recall
        # ===============================
        
        TP_cumsum = torch.cumsum(TP, dim=0)
        FP_cumsum = torch.cumsum(FP, dim=0)

        precision_curve = TP_cumsum / (TP_cumsum + FP_cumsum + eps)
        recall_curve = TP_cumsum / (total_true_bboxes + eps)

        precision_curve = torch.cat([
            torch.tensor([1.0], device=precision_curve.device),
            precision_curve
        ])
        
        recall_curve = torch.cat([
            torch.tensor([0.0], device=precision_curve.device),
            recall_curve
        ])

        precision_curve = torch.flip(
            torch.cummax(torch.flip(precision_curve, [0]), 0)[0],
            [0],
        )

        ap = torch.trapz(precision_curve, recall_curve)
        classes_ap.append(ap)

        tp_sum = TP.sum().item()
        fp_sum = FP.sum().item()

        total_tp += tp_sum
        total_fp += fp_sum

        class_tp[c] = tp_sum
        class_fp[c] = fp_sum
        class_fn[c] = total_true_bboxes - tp_sum

    total_unique_images = len(unique_images)

    return (
        classes_ap,
        class_tp,
        class_fp,
        class_fn,
        total_unique_images,
        images_per_class,
        instances_per_class,
        total_tp,
        total_fp,
        total_gt,
    )

In [13]:
def get_evaluation_bboxes(
    loader,
    model,
    anchors,
    iou_threshold=NMS_IOU_THRESH,
    threshold=MAP_IOU_THRESH,
    box_format="midpoint",
    device="cuda",
    max_boxes=100,
):

    model.eval()

    all_pred_boxes = []
    all_true_boxes = []
    image_idx = 0

    with torch.no_grad():

        for x, labels in tqdm(loader):
            x = x.to(device)
            predictions = model(x)

            batch_size = x.shape[0]

            batch_boxes = [[] for _ in range(batch_size)]
            true_boxes_batch = [[] for _ in range(batch_size)]

            # ===============================
            # 1) PRED + GT DECODE PER SCALE
            # ===============================
            for scale_idx in range(len(predictions)):

                pred_shape = predictions[scale_idx].shape[2]
                anchor = anchors[scale_idx]

                # -------- PREDICTIONS --------
                boxes_scale = cells_to_bboxes(
                    predictions[scale_idx],
                    anchor,
                    S=pred_shape,
                    is_preds=True,
                )

                # -------- GT--------
                label_scale = labels[scale_idx]

                for b_idx in range(batch_size):
                    for a in range(label_scale.shape[1]):
                        for i in range(label_scale.shape[2]):
                            for j in range(label_scale.shape[3]):

                                if label_scale[b_idx, a, i, j, 0] != 1:
                                    continue

                                cls = label_scale[b_idx, a, i, j, 5]

                                bx = label_scale[b_idx, a, i, j, 1]
                                by = label_scale[b_idx, a, i, j, 2]
                                bw = label_scale[b_idx, a, i, j, 3]
                                bh = label_scale[b_idx, a, i, j, 4]

                                true_boxes_batch[b_idx].append([
                                    cls.item(),
                                    1.0,
                                    (bx.item() + j) / pred_shape,
                                    (by.item() + i) / pred_shape,
                                    bw.item() / pred_shape,
                                    bh.item() / pred_shape,
                                ])

                # collect preds
                for b_idx in range(batch_size):
                    batch_boxes[b_idx].extend(boxes_scale[b_idx])

            # ===============================
            # 2) PROCESS EACH IMAGE
            # ===============================
            for b_idx in range(batch_size):

                boxes = batch_boxes[b_idx]

                if len(boxes) == 0:
                    # still add GT
                    for box in true_boxes_batch[b_idx]:
                        all_true_boxes.append([image_idx] + box)
                    image_idx += 1
                    continue

                boxes = torch.tensor(boxes, device=device)

                # confidence filter
                boxes = boxes[boxes[:, 1] > threshold]

                if len(boxes) > 0:

                    # sort by confidence
                    boxes = boxes[boxes[:, 1].argsort(descending=True)]

                    # limit
                    boxes = boxes[:max_boxes]

                    scores = boxes[:, 1]
                    bboxes = boxes[:, 2:6]   # x,y,w,h

                    # ===========================
                    # CLASS-AWARE NMS
                    # ===========================
                    class_ids = boxes[:, 0].long()

                    keep = batched_nms(
                        bboxes,
                        scores,
                        class_ids,
                        iou_threshold,
                    )

                    boxes = boxes[keep]

                    for box in boxes:
                        all_pred_boxes.append([image_idx] + box.tolist())

                # GT append
                for box in true_boxes_batch[b_idx]:
                    all_true_boxes.append([image_idx] + box)

                image_idx += 1

    model.train()

    return all_pred_boxes, all_true_boxes

In [14]:
def cells_to_bboxes(predictions, anchors, S, is_preds=True):
    BATCH_SIZE = predictions.shape[0]
    num_anchors = len(anchors)
    
    box_predictions = predictions[..., 1:5].clone()
    
    if is_preds:
        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        
        box_predictions[..., 0:2] = torch.sigmoid(box_predictions[..., 0:2])
        box_predictions[..., 2:] = torch.exp(box_predictions[..., 2:]) * anchors
        
        scores = torch.sigmoid(predictions[..., 0:1])
        best_class = torch.argmax(predictions[..., 5:], dim=-1).unsqueeze(-1)
    else:
        scores = predictions[..., 0:1]
        best_class = predictions[..., 5:6]

    cell_indices = (
        torch.arange(S)
        .repeat(predictions.shape[0], num_anchors, S, 1)
        .unsqueeze(-1)
        .to(predictions.device)
    )
    
    x = (box_predictions[..., 0:1] + cell_indices) / S
    y = (
        box_predictions[..., 1:2]
        + cell_indices.permute(0,1,3,2,4)
    ) / S
    w_h = box_predictions[..., 2:4] / S
    
    converted_bboxes = torch.cat(
        (best_class, scores, x, y, w_h),
        dim=-1
    ).reshape(BATCH_SIZE, num_anchors * S * S, 6)
    
    return converted_bboxes.tolist()


In [15]:
def check_class_accuracy(model, loader, epoch, threshold, writer):

    model.eval()

    tot_class_preds, correct_class = 0, 0
    tot_noobj, correct_noobj = 0, 0
    tot_obj, correct_obj = 0, 0

    with torch.no_grad():
        for idx, (x, y) in enumerate(tqdm(loader)):

            x = x.to(DEVICE, non_blocking=True)
            out = model(x)

            for i in range(2):
                y[i] = y[i].to(DEVICE, non_blocking=True)

                obj = y[i][..., 0] == 1
                noobj = y[i][..., 0] == 0

                correct_class += (
                    torch.argmax(out[i][..., 5:][obj], dim=-1)
                    == y[i][..., 5][obj]
                ).sum().item()

                tot_class_preds += obj.sum().item()

                obj_preds = torch.sigmoid(out[i][..., 0]) > threshold

                correct_obj += (
                    obj_preds[obj] == y[i][..., 0][obj]
                ).sum().item()

                tot_obj += obj.sum().item()

                correct_noobj += (
                    obj_preds[noobj] == y[i][..., 0][noobj]
                ).sum().item()

                tot_noobj += noobj.sum().item()

    class_acc = (correct_class/(tot_class_preds+1e-16))*100
    no_obj_acc = (correct_noobj/(tot_noobj+1e-16))*100
    obj_acc = (correct_obj/(tot_obj+1e-16))*100

    writer.add_scalar('Class Accuracy/train', class_acc, epoch)
    writer.add_scalar('No Obj Accuracy/train', no_obj_acc, epoch)
    writer.add_scalar('Obj Accuracy/train', obj_acc, epoch)

    print(f"Class accuracy is: {class_acc:.2f}%")
    print(f"No obj accuracy is: {no_obj_acc:.2f}%")
    print(f"Obj accuracy is: {obj_acc:.2f}%")

    model.train()

In [16]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [17]:

def get_loaders(seed=42):
    
    train_dataset = SiStNetDataset(
        transforms=train_transforms,
        S=[RESIZE_TO // 32, RESIZE_TO // 16],
        img_dir=train_data_path,
        label_dir=train_lbl_path,
        anchors=ANCHORS,
    )
     
    valid_dataset = SiStNetDataset(
        transforms=test_transforms,
        S=[RESIZE_TO // 32, RESIZE_TO // 16],
        img_dir=valid_data_path,
        label_dir=valid_lbl_path,
        anchors=ANCHORS,
    )
    
    # ---------- generators ----------
    train_gen = torch.Generator()
    train_gen.manual_seed(seed)

    valid_gen = torch.Generator()
    valid_gen.manual_seed(seed)

    # ---------- loaders ----------
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=4 if NUM_WORKERS > 0 else None,
        worker_init_fn=seed_worker,
        generator=train_gen,  
    )
    
    valid_loader = DataLoader(
        dataset=valid_dataset,
        batch_size=VAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=4 if NUM_WORKERS > 0 else None,
        worker_init_fn=seed_worker,
        generator=valid_gen,    
    )

    return train_loader, valid_loader


In [18]:
def save_checkpoint(model, optimizer, epoch, scheduler, scaler, best_map, seed,
                    filename="./checkpoints/my_checkpoint.pth.tar", message = "Checkpoint saved!"):

    print("###   Saving Checkpoint...")

    checkpoint = {
        "epoch": epoch,
        "seed": seed,
        "best_map": best_map,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler else None,
        "scaler": scaler.state_dict() if scaler else None,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
    }

    torch.save(checkpoint, filename)

    print(message)

In [19]:
def load_full_checkpoint(checkpoint_path, model, optimizer, scheduler, scaler):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)    
    
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    if scheduler and checkpoint["scheduler"]:
        scheduler.load_state_dict(checkpoint["scheduler"])

    if scaler and checkpoint["scaler"]:
        scaler.load_state_dict(checkpoint["scaler"])

    start_epoch = checkpoint["epoch"] + 1
    best_map = checkpoint["best_map"]

    seed = checkpoint["seed"]

    rng_state = torch.ByteTensor(checkpoint["torch_rng_state"])
    cuda_rng_state = torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])
    numpy_rng_state = np.random.set_state(checkpoint["numpy_rng_state"])
    python_rng_state = random.setstate(checkpoint["python_rng_state"])

    print("Full Checkpoint loaded!")
    
    return start_epoch, best_map, seed

In [20]:
class SiStNetDataset(Dataset):
    def __init__(
        self,
        img_dir,
        label_dir,
        anchors,
        S=[13, 26],
        C=20,
        transforms=None,
    ):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*")))

        self.label_paths = [
            os.path.join(label_dir, os.path.basename(p).replace(".png", ".txt"))
            for p in self.img_paths
        ]

        self.transforms = transforms
        self.S = S
        self.C = C

        # anchors (same logic as old)
        self.anchors = torch.tensor(anchors[0] + anchors[1])
        self.num_anchors = self.anchors.shape[0]
        self.num_anchors_per_scale = self.num_anchors // len(S)

        # keep SAME meaning as old code
        self.ignore_iou_thresh = 0.5

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, index):

        # =========================
        # IMAGE
        # =========================
        image = np.array(
            Image.open(self.img_paths[index]).convert("RGB"),
            dtype=np.uint8,
        )

        # =========================
        # LABELS
        # =========================
        boxes = []
        with open(self.label_paths[index]) as f:
            for line in f:
                cls, xc, yc, w, h = map(float, line.split())
                boxes.append([xc, yc, w, h, cls])

        boxes = np.array(boxes)

        # =========================
        # AUGMENTATION
        # =========================
        if self.transforms:
            aug = self.transforms(image=image, bboxes=boxes)
            image = aug["image"]
            bboxes = aug["bboxes"]
        else:
            bboxes = boxes

        # =========================
        # TARGET INITIALIZATION
        # =========================
        targets = [
            torch.zeros(
                (self.num_anchors_per_scale, S, S, 6),
                dtype=torch.float32,
            )
            for S in self.S
        ]

        # =========================
        # ASSIGN LABELS
        # =========================
        for box in bboxes:

            x, y, w, h, cls = box

            # IoU with anchors (UNCHANGED LOGIC)
            iou_anchors = iou_width_height(
                torch.tensor([w, h]),
                self.anchors
            )

            anchor_indices = iou_anchors.argsort(descending=True)

            has_anchor = [False] * len(self.S)

            for anchor_idx in anchor_indices:

                scale_idx = anchor_idx // self.num_anchors_per_scale
                anchor_on_scale = anchor_idx % self.num_anchors_per_scale

                S = self.S[scale_idx]

                # SAFE (prevents out-of-bound crash)
                i = min(S - 1, int(S * y))
                j = min(S - 1, int(S * x))

                anchor_taken = targets[scale_idx][anchor_on_scale, i, j, 0]

                # =========================
                # POSITIVE ASSIGNMENT
                # =========================
                if not anchor_taken and not has_anchor[scale_idx]:

                    targets[scale_idx][anchor_on_scale, i, j, 0] = 1

                    targets[scale_idx][anchor_on_scale, i, j, 1:5] = torch.tensor([
                        S * x - j,
                        S * y - i,
                        w * S,
                        h * S,
                    ])

                    targets[scale_idx][anchor_on_scale, i, j, 5] = int(cls)

                    has_anchor[scale_idx] = True

                elif (
                    not anchor_taken
                    and iou_anchors[anchor_idx] > self.ignore_iou_thresh
                ):
                    targets[scale_idx][anchor_on_scale, i, j, 0] = -1

        return image, tuple(targets)

In [21]:
class SiStNetLoss(nn.Module):
    def __init__(self):
        super().__init__()

        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.entropy = nn.CrossEntropyLoss()
        self.sigmoid = nn.Sigmoid()
 
        self.lambda_noobj = 1
        self.lambda_box = 1
        self.lambda_obj = 1
        self.lambda_class = 1

    def forward(self, predictions, target, anchors):

        obj = target[..., 0] == 1
        noobj = target[..., 0] == 0

        # =========================================================
        # NO OBJECT LOSS
        # =========================================================
        # SAFE improvement: guard empty tensor (does NOT change math)
        if noobj.sum() > 0:
            no_object_loss = self.bce(
                predictions[..., 0:1][noobj],
                target[..., 0:1][noobj],
            )
        else:
            no_object_loss = torch.tensor(
                0.0, device=predictions.device
            )

        # =========================================================
        # OBJECT LOSS
        # =========================================================
        anchors = anchors.reshape(1, 3, 1, 1, 2)

        box_preds = torch.cat(
            [
                self.sigmoid(predictions[..., 1:3]),
                torch.exp(predictions[..., 3:5]) * anchors,
            ],
            dim=-1,
        )

        ious = intersection_over_union(
            box_preds[obj],
            target[..., 1:5][obj],
        ).detach()

        if obj.sum() > 0:
            object_loss = self.mse(
                self.sigmoid(predictions[..., 0:1][obj]),
                ious * target[..., 0:1][obj],
            )
        else:
            object_loss = torch.tensor(
                0.0, device=predictions.device
            )

        # =========================================================
        # BOX LOSS
        # =========================================================
        # IMPORTANT: clone prevents in-place gradient side effects
        pred_boxes = predictions[..., 1:5].clone()
        target_boxes = target[..., 1:5].clone()

        pred_boxes[..., 0:2] = self.sigmoid(pred_boxes[..., 0:2])

        target_boxes[..., 2:4] = torch.log(
            1e-16 + target_boxes[..., 2:4] / anchors
        )

        if obj.sum() > 0:
            box_loss = self.mse(
                pred_boxes[obj],
                target_boxes[obj],
            )
        else:
            box_loss = torch.tensor(
                0.0, device=predictions.device
            )

        # =========================================================
        # CLASS LOSS
        # =========================================================
        if obj.sum() > 0:
            class_loss = self.entropy(
                predictions[..., 5:][obj],
                target[..., 5][obj].long(),
            )
        else:
            class_loss = torch.tensor(
                0.0, device=predictions.device
            )

        # =========================================================
        # TOTAL LOSS
        # =========================================================
        loss = (
            self.lambda_box * box_loss
            + self.lambda_obj * object_loss
            + self.lambda_noobj * no_object_loss
            + self.lambda_class * class_loss
        )

        pos_ratio = obj.float().mean()

        return loss, {
            "box": box_loss,
            "obj": object_loss,
            "noobj": no_object_loss,
            "class": class_loss,
            "mean_iou": ious.mean()
            if obj.sum() > 0
            else torch.tensor(0.0, device=predictions.device),
            "pos_ratio": pos_ratio,
        }

In [22]:
def safe_float(x):
    if torch.is_tensor(x):
        return x.item()
    if isinstance(x, (np.floating, np.ndarray)):
        return float(x)
    return float(x)

In [23]:
@torch.inference_mode()
def evaluate_fn(model, valid_loader, scaled_anchor, epoch, writer):

    model.eval()

    check_class_accuracy(
        model,
        valid_loader,
        epoch,
        threshold=CONF_THRESHOLD,
        writer=writer
    )

    torch.cuda.synchronize()
    t0 = time.time()

    pred_boxes, true_boxes = get_evaluation_bboxes(
        valid_loader,
        model,
        anchors=scaled_anchor,
        iou_threshold=NMS_IOU_THRESH,
        threshold=MAP_IOU_THRESH,
    )

    (
        classes_ap,
        class_tp,
        class_fp,
        class_fn,
        total_images,
        images_per_class,
        instances_per_class,
        total_tp,
        total_fp,
        total_gt,
    ) = mean_average_precision(
        pred_boxes,
        true_boxes,
        epoch,
        num_classes=NUM_CLASSES,
        iou_threshold=MAP_IOU_THRESH,
        conf_threshold=CONF_THRESHOLD,
    )

    torch.cuda.synchronize()
    print("mAP TIME:", time.time() - t0)

    metrics = compute_metrics(
        class_tp,
        class_fp,
        class_fn,
        total_images,
        images_per_class,
        instances_per_class,
        total_tp,
        total_fp,
        total_gt,
        classes_ap,
    )

    ap_list = metrics["AP_per_class"]

    if len(ap_list) > 0:
        mapval = float(
            torch.mean(
                torch.tensor(
                    [
                        float(x.item() if torch.is_tensor(x) else x)
                        for x in ap_list
                    ]
                )
            )
        )
    else:
        mapval = torch.tensor(0.0)

    return mapval, metrics

In [24]:
@torch.no_grad()
def evaluate_loss(loader, model, loss_fn, anchors):
    model.eval()

    total_loss = 0.0
    count = 0

    for x, y in loader:

        x = x.to(DEVICE, non_blocking=True)
        y0 = y[0].to(DEVICE, non_blocking=True)
        y1 = y[1].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=AMP):

            out = model(x)

            loss0, _ = loss_fn(out[0], y0, anchors[0])
            loss1, _ = loss_fn(out[1], y1, anchors[1])

            loss = loss0 + loss1

        total_loss += loss.item()
        count += 1

    return total_loss / max(count, 1)

In [25]:
def log_metrics(writer, epoch, metrics, mapval):
 
    for i, cls in enumerate(CLASSES):

        ap = safe_float(metrics["AP_per_class"][i])

        if torch.is_tensor(ap):
            ap = ap.item()
        ap = float(ap)

        writer.add_scalar(f"mAP/{cls}", ap, epoch)
 
    writer.add_scalar("mAP/all", float(mapval), epoch)

    writer.add_scalar("precision", float(metrics["precision"]), epoch)
    writer.add_scalar("recall", float(metrics["recall"]), epoch)
    writer.add_scalar("f1", float(metrics["f1"]), epoch)

In [26]:
def compute_metrics(
    class_tp,
    class_fp,
    class_fn,
    total_images,
    images_per_class,
    instances_per_class,
    total_tp,
    total_fp,
    total_gt,
    classes_ap,
    eps=1e-9,
):

    total_tp = float(total_tp)
    total_fp = float(total_fp)
    total_gt = float(total_gt)

    precision = total_tp / (total_tp + total_fp + eps)
    recall = total_tp / (total_gt + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)

    precision_per_class = class_tp / (class_tp + class_fp + eps)
    recall_per_class = class_tp / (class_tp + class_fn + eps)

    f1_per_class = (
        2 * precision_per_class * recall_per_class
        / (precision_per_class + recall_per_class + eps)
    )

    average_recall = recall_per_class.mean()

    FN = total_gt - total_tp

    return {
        "AP_per_class": classes_ap,

        "precision": precision,
        "recall": recall,
        "f1": f1,

        "total_images": total_images,
        "precision_per_class": precision_per_class.tolist(),
        "recall_per_class": recall_per_class.tolist(),
        "f1_per_class": f1_per_class.tolist(),

        "tp_per_class": class_tp.tolist(),
        "fp_per_class": class_fp.tolist(),
        "fn_per_class": class_fn.tolist(),

        "images_per_class": images_per_class.tolist(),
        "instances_per_class": instances_per_class.tolist(),

        "average_recall": float(average_recall.item() if torch.is_tensor(average_recall) else average_recall),

        "fp": float(total_fp),
        "fn": float(FN),
    }

In [27]:
def log_loss(writer, loss, scaled_loss, loss_dict, optimizer_step):

    writer.add_scalar("loss/box", loss_dict["box"].item(), optimizer_step)
    writer.add_scalar("loss/obj", loss_dict["obj"].item(), optimizer_step)
    writer.add_scalar("loss/noobj", loss_dict["noobj"].item(), optimizer_step)
    writer.add_scalar("loss/class", loss_dict["class"].item(), optimizer_step)

    writer.add_scalar("metric/mean_iou", loss_dict["mean_iou"].item(), optimizer_step)
    writer.add_scalar("metric/pos_ratio", loss_dict["pos_ratio"].item(), optimizer_step)

    writer.add_scalar("Loss/train", loss.item(), optimizer_step)
    writer.add_scalar("Loss/scaled_train", scaled_loss.item(), optimizer_step)

In [28]:
def train_fn(
    train_loader,
    model,
    epoch,
    optimizer,
    loss_fn,
    scaler,
    anchors,
    writer,
):

    model.train()

    loop = tqdm(train_loader, leave=True)

    optimizer_step = 0

    running_loss = 0.0
    running_step_loss = 0.0

    batch_count = 0
    step_count = 0

    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (x, y) in enumerate(loop):

        x = x.to(DEVICE, non_blocking=True)
        y0 = y[0].to(DEVICE, non_blocking=True)
        y1 = y[1].to(DEVICE, non_blocking=True)

        # ---------------- AMP ----------------
        with torch.autocast("cuda", enabled=AMP):

            out = model(x)

            loss0, dict0 = loss_fn(out[0], y0, anchors[0])
            loss1, dict1 = loss_fn(out[1], y1, anchors[1])

            loss = loss0 + loss1

            loss_dict = {
                "box": (dict0["box"] + dict1["box"]).detach().cpu(),
                "obj": (dict0["obj"] + dict1["obj"]).detach().cpu(),
                "noobj": (dict0["noobj"] + dict1["noobj"]).detach().cpu(),
                "class": (dict0["class"] + dict1["class"]).detach().cpu(),
                "mean_iou": (dict0["mean_iou"] + dict1["mean_iou"]).detach().cpu(),
                "pos_ratio": (dict0["pos_ratio"] + dict1["pos_ratio"]).detach().cpu(),
            }

            scaled_loss = loss / ACCUMULATE

        # ---------------- BACKWARD ----------------
        scaler.scale(scaled_loss).backward()

        is_last = (batch_idx + 1) == len(train_loader)
        do_step = ((batch_idx + 1) % ACCUMULATE == 0) or is_last

        if do_step:

            scaler.unscale_(optimizer)

            total_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                10.0
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

            optimizer_step += 1

            writer.add_scalar("grad_norm", total_norm, optimizer_step)

            step_loss = running_step_loss / max(step_count, 1)
            writer.add_scalar("Loss/optimizer_step", step_loss, optimizer_step)

            # reset step loss tracking
            running_step_loss = 0.0
            step_count = 0

        # ---------------- batch loss ----------------
        running_loss += loss.item()
        running_step_loss += loss.item()
        batch_count += 1
        step_count += 1

        mean_loss = running_loss / batch_count

        loop.set_postfix(loss=f"{mean_loss:.4f}")

        log_loss(
            writer,
            loss,
            scaled_loss,
            loss_dict,
            optimizer_step
        )

    writer.add_scalar("metric/loss_smooth", mean_loss, epoch)
    writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

    return mean_loss

In [29]:
def train_one_run(seed):

    seed_everything(seed)

    with SummaryWriter(log_dir=f"runs/seed_{seed}") as writer:

        best_map = -float("inf")
        counter = 0
        start_epoch = 0

        model = SiStNet(num_classes=NUM_CLASSES).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=EPOCHS
        )

        loss_fn = SiStNetLoss()

        scaler = torch.cuda.amp.GradScaler(
            enabled=AMP,
            init_scale=2**12
        )

        # ---------------- checkpoint load ----------------
        if LOAD_MODEL:
            start_epoch, best_map, _ = load_full_checkpoint(
                CHECKPOINT_FILE,
                model,
                optimizer,
                scheduler,
                scaler
            )

            best_map = float(best_map)

        train_loader, valid_loader = get_loaders(seed)

        # ---------------- anchors ----------------
        scaled_anchors = [
            torch.tensor(ANCHORS[i], device=DEVICE, dtype=torch.float32) * float(S[i])
            for i in range(len(S))
        ]

        # ================= TRAIN LOOP =================
        for epoch in range(start_epoch, EPOCHS):

            model.train()

            current_epoch = epoch + 1

            print("-" * 85)
            print(f" Epoch: {current_epoch}/{EPOCHS}")

            # ---------------- warmup + cosine ----------------
            if epoch < WARMUP_EPOCHS:
                lr = BASE_LR * current_epoch / WARMUP_EPOCHS
                for g in optimizer.param_groups:
                    g["lr"] = lr
            else:
                scheduler.step()
                lr = optimizer.param_groups[0]["lr"]

            # ---------------- train ----------------
            train_loss = train_fn(
                train_loader,
                model,
                epoch,
                optimizer,
                loss_fn,
                scaler,
                scaled_anchors,
                writer
            )

            # ---------------- val loss ----------------
            val_loss = evaluate_loss(
                valid_loader,
                model,
                loss_fn,
                scaled_anchors,
            )

            writer.add_scalar("Loss/val_epoch", val_loss, epoch)

            writer.add_scalar(
                "gap/train_val_loss",
                train_loss - val_loss,
                epoch
            )

            # ---------------- eval ----------------
            do_eval = (
                current_epoch == 1
                or current_epoch == 15
                or current_epoch > 24
            )

            if do_eval:

                mapval, metrics = evaluate_fn(
                    model,
                    valid_loader,
                    scaled_anchors,
                    epoch,
                    writer
                )

                mapval = float(mapval)

                precision = metrics["precision"]
                recall = metrics["recall"]
                f1 = metrics["f1"]
                fp = metrics["fp"]
                fn = metrics["fn"]

                # ---------------- best model ----------------
                if mapval > best_map:
                    best_map = mapval
                    counter = 0

                    if SAVE_MODEL:
                        save_checkpoint(
                            model,
                            optimizer,
                            epoch,
                            scheduler,
                            scaler,
                            best_map,
                            seed,
                            filename=f"./checkpoints/best_lw_{seed}.pth.tar",
                            message = "Best checkpoint saved!"
                        )

                else:
                    counter += 1

                # ---------------- ALWAYS save last ----------------
                save_checkpoint(
                    model,
                    optimizer,
                    epoch,
                    scheduler,
                    scaler,
                    best_map,
                    seed,
                    filename=f"./checkpoints/last_lw_checkpoint_{seed}.pth.tar",
                    message = "Last checkpoint saved!"
                )

                # ---------------- logging ----------------
                print(f"{'Class':15}{'Images':10}{'Images/Classes':10}{'Instances':15}{'P':10}{'R':10}{'F1':10}{'mAP':15}{'FP':10}{'FN':10}")

                print("-" * 85)

                print(
                    f"{'all':15}"
                    f"{metrics['total_images']:10}"
                    f"{int(sum(metrics['images_per_class'])):10}"
                    f"{int(sum(metrics['instances_per_class'])):15}"
                    f"{precision:<10.3f}"
                    f"{recall:<10.3f}"
                    f"{f1:<10.3f}"
                    f"{mapval:<15.3f}"
                    f"{int(fp):10}"
                    f"{int(fn):10}"
                )
                
                print("-" * 85)

                for i in range(NUM_CLASSES):
                    print(f"{CLASSES[i]:15}"
                          f"{metrics['total_images']:10}"
                          f"{int(metrics['images_per_class'][i]):10}"
                          f"{int(metrics['instances_per_class'][i]):15}"
                          f"{metrics['precision_per_class'][i]:10.3f}"
                          f"{metrics['recall_per_class'][i]:10.3f}"
                          f"{metrics['f1_per_class'][i]:10.3f}"
                          f"{metrics['AP_per_class'][i]:15.3f}"
                          f"{int(metrics['fp_per_class'][i]):10}"
                          f"{int(metrics['fn_per_class'][i]):10}")

                log_metrics(writer, epoch, metrics, mapval)

                print("\n===== DATASET SANITY CHECK =====")
                print(f"Unique GT images: {metrics['total_images']}")
                print(f"Sum images_per_class: {int(sum(metrics['images_per_class']))}")
                print(f"Total instances: {int(sum(metrics['instances_per_class']))}")
                print("================================\n")

                model.train()

        return best_map

In [30]:
def main():
    SEEDS = [42, 123, 999]
    all_maps = []

    for seed in SEEDS:
        print(f"\n===== RUN WITH SEED {seed} =====")

        final_map = train_one_run(seed)
        all_maps.append(float(final_map))

    mean_map = np.mean(all_maps)
    std_map = np.std(all_maps)

    print("=-" * 85)
    print("FINAL RESULT :")
    print(f"mAP50 = {mean_map:.3f} ± {std_map:.3f}")


if __name__ == "__main__":
    main()


===== RUN WITH SEED 42 =====
-------------------------------------------------------------------------------------
 Epoch: 1/30


100%|███████████████████████████████████████████| 30/30 [00:13<00:00,  2.20it/s]


Class accuracy is: 73.64%
No obj accuracy is: 99.49%
Obj accuracy is: 14.53%


100%|███████████████████████████████████████████| 30/30 [00:42<00:00,  1.41s/it]


mAP TIME: 46.832733392715454
###   Saving Checkpoint...
Best checkpoint saved!
###   Saving Checkpoint...
Last checkpoint saved!
Class          Images    Images/ClassesInstances      P         R         F1        mAP            FP        FN        
-------------------------------------------------------------------------------------
all                   296       534           30770.032     0.097     0.048     0.001                9128      2777
-------------------------------------------------------------------------------------
Car                   296       267           2274     0.032     0.132     0.051          0.006      9128      1974
Van                   296        81            221     0.000     0.000     0.000          0.000         0       221
Truck                 296        34             68     0.000     0.000     0.000          0.000         0        68
Pedestrian            296        64            272     0.000     0.000     0.000          0.000         0       272

...

mAP TIME: 47.8423593044281
###   Saving Checkpoint...
Last checkpoint saved!
Class          Images    Images/ClassesInstances      P         R         F1        mAP            FP        FN        
-------------------------------------------------------------------------------------
all                   296       534           30770.080     0.321     0.128     0.046               11390      2090
-------------------------------------------------------------------------------------
Car                   296       267           2274     0.089     0.377     0.144          0.071      8753      1416
Van                   296        81            221     0.053     0.208     0.085          0.033       816       175
Truck                 296        34             68     0.200     0.265     0.228          0.216        72        50
Pedestrian            296        64            272     0.037     0.184     0.062          0.013      1287       222
Person_sitting        296         3             14 

100%|███████████████████████████████████████████| 30/30 [00:13<00:00,  2.19it/s]


Class accuracy is: 85.70%
No obj accuracy is: 99.38%
Obj accuracy is: 55.67%


100%|███████████████████████████████████████████| 30/30 [00:41<00:00,  1.39s/it]


mAP TIME: 47.41733169555664
###   Saving Checkpoint...
Last checkpoint saved!
Class          Images    Images/ClassesInstances      P         R         F1        mAP            FP        FN        
-------------------------------------------------------------------------------------
all                   296       534           30770.081     0.320     0.129     0.046               11258      2091
-------------------------------------------------------------------------------------
Car                   296       267           2274     0.089     0.376     0.144          0.071      8709      1419
Van                   296        81            221     0.060     0.222     0.094          0.036       774       172
Truck                 296        34             68     0.218     0.279     0.245          0.221        68        49
Pedestrian            296        64            272     0.034     0.173     0.057          0.012      1333       225
Person_sitting        296         3             14

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=runs